# D65-FairFace7-ROI Walkthrough
Chart-free flash/no-flash cheek colorimetry with **FairFace-7 ROI sampling**.

**Pipeline (claimable color + deployment ROI):**
1. Demosaic DNGs **without** camera WB → reflectance \(R_0=\sqrt{A_0\odot B_0'}\)
2. Apple Vision cheek mask (from zip landmarks — **no MediaPipe**)
3. Affine RGB→XYZ + Bradford CAT **5500 K → D65**
4. FairFace-7 on an 8-bit face crop → `specular_tone` cheek sampling

| Path | Typical mean ΔE₀₀ |
|---|---:|
| Frozen trimmed mean (no ROI) | ~5.55 |
| **D65-FairFace7-ROI** | **~3.63** |
| Old MediaPipe cheek (reference) | ~6.36 |

### Colab steps
1. Runtime → GPU (optional)
2. Run cells **top to bottom**
3. Cell 2 pulls Pansor zips from the shared Drive folder via API (Shared-with-me is not visible to the mount)
4. FitSkin Inside Lab is **hardcoded** (no demographics xlsx required)

Shared folder: https://drive.google.com/drive/folders/1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep


## 0 — Setup


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q rawpy opencv-python-headless numpy matplotlib openpyxl gdown
try:
    import torch, torchvision  # noqa: F401
except ImportError:
    !pip install -q torch torchvision

import os, sys, json, subprocess
from pathlib import Path

REPO_URL = "https://github.com/RooneyEmily/Fitskin.git"
if os.path.isdir("Fitskin"):
    !cd Fitskin && git pull --ff-only || true
else:
    !git clone {REPO_URL}

REPO = Path("Fitskin").resolve()
assert (REPO / "models" / "fairface_race.py").is_file(), "Fitskin clone missing models/ — check git pull / PAT"
assert (REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py").is_file()

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

sys.path = [str(REPO)] + [p for p in sys.path if Path(p).resolve() != REPO]

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

from delta_e_2000 import delta_e_2000
from flash_noflash_spectral import planck_xyz_y1
from models.fairface_race import FairFacePredictor, face_rgb_crop_from_landmarks
from scripts.evaluate_pansor20_chartfree_d65 import (
    D65,
    bradford_cat_matrix,
    linear_rgb_to_preview_bgr,
    load_dng_linear,
    load_apple_landmarks,
    apple_face_cheek_masks,
    match_flash_exposure,
    load_affine,
    discover_indoor_trials,
    extract_zip,
    mean_lab_on_mask,
)

CAL_DIR = REPO / "calibration" / "tier3_affine"
FAIRFACE_DIR = REPO / "calibration" / "fairface"
FAIRFACE_DIR.mkdir(parents=True, exist_ok=True)

# Local runtime copy of Pansor (filled by Cell 2 via Drive API)
PANSOR_ROOT = Path("/content/Pansor Dataset")
PANSOR_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path("/content/pansor_extract")
OUT_DIR = Path("/content/d65_fairface7_roi_results")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

FF7 = FAIRFACE_DIR / "res34_fair_align_multi_7_20190809.pt"
if not FF7.is_file():
    print("Downloading FairFace-7 weights (~82 MB)…")
    !gdown 11y0Wi3YQf21a_VcspUV4FwqzhMcfaVAB -O "{FF7}"
assert FF7.is_file()

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("REPO:", REPO)
print("PANSOR_ROOT:", PANSOR_ROOT)
print("Setup OK.")


## 1 — Download Pansor zips (Drive API)

Colab’s Drive **mount does not see “Shared with me”**. This cell copies files by folder ID into `/content/Pansor Dataset`.

Use the **same Google account** that can open the shared FitSkin folder.  
`LIMIT_PARTICIPANTS = 0` downloads **all** participants (needed for ethnicity plots; ~4.7 GB). Set `1` for a fast Bryan-only demo.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Drive API: copy Participant zips → /content/Pansor Dataset
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

PANSOR_FOLDER_ID = "1RqbqHzTiezUAwlm9xRON0dbZAcDXn9Ep"
LIMIT_PARTICIPANTS = 0  # 0 = all participants (~4.7 GB); 1 = Bryan only (fast demo)

drive = build("drive", "v3")
FOLDER_MIME = "application/vnd.google-apps.folder"

def list_children(folder_id):
    out, token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=token, pageSize=1000,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        out.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return out

def download_file(file_id, dest: Path):
    if dest.is_file() and dest.stat().st_size > 0:
        print("exists", dest.name)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            status, done = dl.next_chunk()
    print("saved", dest)

meta = drive.files().get(fileId=PANSOR_FOLDER_ID, fields="name", supportsAllDrives=True).execute()
print("Shared folder:", meta["name"])

children = list_children(PANSOR_FOLDER_ID)
print("Root:", sorted(f["name"] for f in children)[:20])

# Shared link may be a parent that contains "Pansor Dataset"
nested = [
    f for f in children
    if f["mimeType"] == FOLDER_MIME and f["name"].strip().lower() == "pansor dataset"
]
if nested:
    children = list_children(nested[0]["id"])
    print("Entered nested Pansor Dataset:", sorted(f["name"] for f in children)[:20])

parts = sorted(
    [f for f in children if f["mimeType"] == FOLDER_MIME and f["name"].startswith("Participant")],
    key=lambda x: x["name"],
)
assert parts, "No Participant folders — wrong Google account or folder ID?"
print(f"Participant folders: {len(parts)}")

todo = parts if not LIMIT_PARTICIPANTS else parts[:LIMIT_PARTICIPANTS]
for pf in todo:
    zips = [f for f in list_children(pf["id"]) if f["name"].lower().endswith(".zip")]
    # Prefer indoor chart-free zips (no bag/outside/light in name)
    indoor = [
        z for z in zips
        if "bag" not in z["name"].lower()
        and "outside" not in z["name"].lower()
        and "light" not in z["name"].lower()
    ]
    use = indoor or zips
    print(f"{pf['name']}: downloading {len(use)} zip(s)")
    for z in use:
        download_file(z["id"], PANSOR_ROOT / pf["name"] / z["name"])

print("Zips on disk:", list(PANSOR_ROOT.glob("Participant */*.zip")))


## 2 — Calibration, FairFace, demographics


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Affine + CAT + FairFace-7 + hardcoded FitSkin Inside Lab
# ══════════════════════════════════════════════════════════════════════════════
M = load_affine(CAL_DIR)
xyz_w = planck_xyz_y1(5500.0, 0.0)
CAT = bradford_cat_matrix(xyz_w, D65)
print("Affine M shape:", M.shape)
print("W_5500 XYZ:", np.round(xyz_w, 4))
print("D65 XYZ:", D65)

ff = FairFacePredictor.load(mode="7", weights_dir=FAIRFACE_DIR)
print("FairFace device:", ff.device)

# Inside Lab from Pansor Dataset Demographics.xlsx (hardcoded — no Drive xlsx needed)
demo = {
    1: {"name": "Bryan", "ethnicity": "Black", "fitskin_L": 27.68, "fitskin_a": 8.79, "fitskin_b": 13.84},
    2: {"name": "Dylan", "ethnicity": "White", "fitskin_L": 61.62, "fitskin_a": 13.42, "fitskin_b": 16.5},
    3: {"name": "Eric", "ethnicity": "Asian", "fitskin_L": 55.51, "fitskin_a": 12.75, "fitskin_b": 19.13},
    4: {"name": "Luca", "ethnicity": "White", "fitskin_L": 64.62, "fitskin_a": 11.13, "fitskin_b": 16.66},
    5: {"name": "Ray", "ethnicity": "Indian", "fitskin_L": 59.63, "fitskin_a": 11.25, "fitskin_b": 17.32},
    6: {"name": "Shuyi", "ethnicity": "Asian", "fitskin_L": 59.02, "fitskin_a": 12.56, "fitskin_b": 21.19},
    7: {"name": "Sonali", "ethnicity": "Indian", "fitskin_L": 52.84, "fitskin_a": 10.91, "fitskin_b": 22.05},
    8: {"name": "Utsav", "ethnicity": "Indian", "fitskin_L": 49.65, "fitskin_a": 13.08, "fitskin_b": 21.9},
    9: {"name": "Vivian", "ethnicity": "Black", "fitskin_L": 29.64, "fitskin_a": 10.23, "fitskin_b": 15.54},
    10: {"name": "Yuan", "ethnicity": "Asian", "fitskin_L": 68.0, "fitskin_a": 9.99, "fitskin_b": 15.59},
    11: {"name": "Zeevan", "ethnicity": "Indian", "fitskin_L": 50.25, "fitskin_a": 13.49, "fitskin_b": 23.47},
    12: {"name": "Sanaz", "ethnicity": "Iranian", "fitskin_L": 58.85, "fitskin_a": 12.17, "fitskin_b": 19.64},
    13: {"name": "Brendan", "ethnicity": "White", "fitskin_L": 60.53, "fitskin_a": 14.55, "fitskin_b": 17.56},
    14: {"name": "Raees", "ethnicity": "Indian", "fitskin_L": 48.47, "fitskin_a": 12.17, "fitskin_b": 21.97},
    15: {"name": "Nima", "ethnicity": "Iranian", "fitskin_L": 58.01, "fitskin_a": 12.11, "fitskin_b": 18.26},
    16: {"name": "Nick", "ethnicity": "White", "fitskin_L": 63.14, "fitskin_a": 12.31, "fitskin_b": 17.27},
    17: {"name": "Jalona", "ethnicity": "Black", "fitskin_L": 41.64, "fitskin_a": 11.72, "fitskin_b": 21.64},
    18: {"name": "Gabe", "ethnicity": "White", "fitskin_L": 65.26, "fitskin_a": 11.88, "fitskin_b": 15.65},
    19: {"name": "Chidera", "ethnicity": "Black", "fitskin_L": 29.64, "fitskin_a": 10.23, "fitskin_b": 15.45},
    20: {"name": "Charles", "ethnicity": "Asian", "fitskin_L": 59.82, "fitskin_a": 11.59, "fitskin_b": 21.14},
}
print(f"Demographics: {len(demo)} participants")
print("Example P1:", demo[1])
print("Zips available:", len(list(PANSOR_ROOT.glob("Participant */*.zip"))))


## 3 — Single-trial walkthrough


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Pick one indoor trial and unpack
# ══════════════════════════════════════════════════════════════════════════════
trials = discover_indoor_trials(PANSOR_ROOT)
print(f"Indoor chart-free trials found: {len(trials)}")
assert trials, "No zips — re-run Cell 2 (Drive API download)"

PARTICIPANT_ID = int(trials[0]["participant_id"])
TRIAL = int(trials[0]["trial"])
# PARTICIPANT_ID, TRIAL = 6, 1  # Shuyi
# PARTICIPANT_ID, TRIAL = 1, 1  # Bryan

t = next(x for x in trials if int(x["participant_id"]) == PARTICIPANT_ID and int(x["trial"]) == TRIAL)
meta = demo[PARTICIPANT_ID]
print("Selected:", t["subject_id"], meta["name"], meta["ethnicity"])
print("Zip:", t["zip_path"])
print("FitSkin Lab:", meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"])

nf, fl, lm_path = extract_zip(Path(t["zip_path"]), WORK_DIR / t["subject_id"])
print("Extracted:", nf.name, "|", fl.name, "|", lm_path.name)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — RAW demosaic → Apple cheek mask → reflectance R0
# Lab uses linear RAW. Preview is 8-bit stretch for display / FairFace only.
# ══════════════════════════════════════════════════════════════════════════════
A0 = load_dng_linear(nf, half_size=True, use_camera_wb=False)
B0 = load_dng_linear(fl, half_size=True, use_camera_wb=False)
if B0.shape != A0.shape:
    B0 = cv2.resize(B0, (A0.shape[1], A0.shape[0]), interpolation=cv2.INTER_AREA)

lm = load_apple_landmarks(lm_path)
_, cheek = apple_face_cheek_masks(lm, A0.shape[0], A0.shape[1])
print("Shape:", A0.shape, "| cheek px:", int(np.count_nonzero(cheek)))

B0m, s = match_flash_exposure(A0, B0, cheek)
R0 = np.sqrt(np.maximum(A0, 0) * np.maximum(B0m, 0) + 1e-8)
print(f"Flash scale s={s:.4f}")

preview = linear_rgb_to_preview_bgr(A0)
overlay = preview.copy()
overlay[cheek > 0] = (0.6 * overlay[cheek > 0] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)); ax[0].set_title("No-flash (from RAW)"); ax[0].axis("off")
ax[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); ax[1].set_title("Apple cheek mask"); ax[1].axis("off")
ax[2].imshow(np.clip(R0 ** (1 / 2.2), 0, 1)); ax[2].set_title("Reflectance R0"); ax[2].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Frozen color path (trimmed mean, no FairFace)
# ══════════════════════════════════════════════════════════════════════════════
fit = np.array([meta["fitskin_L"], meta["fitskin_a"], meta["fitskin_b"]], dtype=np.float64)
Lab_frozen, _ = mean_lab_on_mask(
    R0, cheek, M, xyz_scene_white=xyz_w, cat_degree=1.0, l_sampling="off",
)
de_frozen = float(delta_e_2000(Lab_frozen, fit))
print(f"Frozen Lab = ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})")
print(f"FitSkin    = ({fit[0]:.1f}, {fit[1]:.1f}, {fit[2]:.1f})")
print(f"ΔE00 frozen = {de_frozen:.2f}  (cohort mean ~5.55)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — FairFace-7 prior → specular_tone ROI (deployment path)
# ══════════════════════════════════════════════════════════════════════════════
face_rgb = face_rgb_crop_from_landmarks(preview, lm, padding=0.35)
ff_out = ff.predict_rgb(face_rgb)
print("FairFace-7:", ff_out["fairface_label"], f"(conf={ff_out['confidence']:.2f})")
print("→ ROI key:", ff_out["predicted_ethnicity"])

plt.figure(figsize=(3, 3))
plt.imshow(face_rgb); plt.title(ff_out["fairface_label"]); plt.axis("off"); plt.show()

Lab_ff, sm = mean_lab_on_mask(
    R0, cheek, M, xyz_scene_white=xyz_w, cat_degree=1.0,
    l_sampling="specular_tone", ethnicity=ff_out["predicted_ethnicity"],
)
de_ff = float(delta_e_2000(Lab_ff, fit))

Lab_oracle, _ = mean_lab_on_mask(
    R0, cheek, M, xyz_scene_white=xyz_w, cat_degree=1.0,
    l_sampling="specular_tone", ethnicity=meta["ethnicity"],
)
de_oracle = float(delta_e_2000(Lab_oracle, fit))

print(f"D65-FairFace7-ROI = ({Lab_ff[0]:.1f}, {Lab_ff[1]:.1f}, {Lab_ff[2]:.1f})  ΔE00={de_ff:.2f}")
print(f"Oracle ethnicity  = ({Lab_oracle[0]:.1f}, {Lab_oracle[1]:.1f}, {Lab_oracle[2]:.1f})  ΔE00={de_oracle:.2f}")
print(f"Frozen            = ({Lab_frozen[0]:.1f}, {Lab_frozen[1]:.1f}, {Lab_frozen[2]:.1f})  ΔE00={de_frozen:.2f}")
print("Sampling:", sm)


## 4 — Cohort eval + plots

Runs the production evaluator on whatever zips Cell 2 downloaded, then plots ΔE₀₀.
With all participants downloaded you get ethnicity histograms (Black/Indian/Asian/Iranian/White). `LIMIT_PARTICIPANTS=1` is Bryan-only.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Cohort eval (writes demog xlsx from hardcoded demo, then CLI)
# ══════════════════════════════════════════════════════════════════════════════
from openpyxl import Workbook

DEMOG_XLSX = PANSOR_ROOT / "Pansor Dataset Demographics.xlsx"
wb = Workbook()
ws = wb.active
ws.append(["Pansor Dataset Demographics"])
ws.append(["Participant ID", "Name", "Ethnicity", "L*", "a*", "b*"])
for pid, m in sorted(demo.items()):
    ws.append([pid, m["name"], m["ethnicity"], m["fitskin_L"], m["fitskin_a"], m["fitskin_b"]])
wb.save(DEMOG_XLSX)

n_zips = len(list(PANSOR_ROOT.glob("Participant */*.zip")))
print("Wrote", DEMOG_XLSX)
print("Zips available:", n_zips)
assert n_zips > 0, "No zips — run Cell 2 first"

cmd = [
    "python3", str(REPO / "scripts" / "evaluate_pansor20_chartfree_d65.py"),
    "--data-root", str(PANSOR_ROOT),
    "--demographics", str(DEMOG_XLSX),
    "--cal-dir", str(CAL_DIR),
    "--scr-mode", "preawb_cat",
    "--fixed-cat-k", "5500",
    "--l-sampling", "fairface7",
    "--fairface-dir", str(FAIRFACE_DIR),
    "--emily-tsv",
    "--work-dir", str(WORK_DIR),
    "--out-dir", str(OUT_DIR),
]
print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"evaluator failed ({proc.returncode})")

summary = json.loads((OUT_DIR / "summary.json").read_text())
print(f"n={summary['n_trials']}  mean={summary['mean_de00']:.2f}  median={summary['median_de00']:.2f}")
for eth, st in summary.get("by_ethnicity", {}).items():
    print(f"  {eth:10s} n={st['n']:3d}  mean={st['mean_de00']:.2f}  med={st['median_de00']:.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Plot results (works for n=4 Bryan-only or full cohort)
# ══════════════════════════════════════════════════════════════════════════════
import csv
from statistics import mean, median

csv_path = OUT_DIR / "pansor20_chartfree_d65.csv"
if not csv_path.is_file():
    hits = list(OUT_DIR.glob("*.csv"))
    assert hits, f"No CSV in {OUT_DIR}"
    csv_path = hits[0]

rows = list(csv.DictReader(csv_path.open()))
print(f"Loaded {len(rows)} trials from {csv_path.name}")

# --- Per-trial bar chart (always useful; shows your 4 results clearly) ---
labels = [r["subject_id"] for r in rows]
vals = [float(r["de00"]) for r in rows]
eths = [r.get("ethnicity", "") for r in rows]

fig, ax = plt.subplots(figsize=(max(7, 0.55 * len(rows) + 2), 3.8))
colors_map = {
    "Black": "#2c3e50", "Indian": "#c0392b", "Asian": "#2980b9",
    "Iranian": "#16a085", "White": "#d4a017",
}
bar_colors = [colors_map.get(e, "#7f8c8d") for e in eths]
ax.bar(labels, vals, color=bar_colors, edgecolor="white")
ax.axhline(mean(vals), color="#e74c3c", ls="--", label=f"mean={mean(vals):.2f}")
ax.axhline(5.55, color="#27ae60", ls=":", label="frozen ref 5.55")
ax.axhline(5.93, color="#8e44ad", ls=":", alpha=0.7, label="camera-WB ref 5.93")
ax.set_ylabel(r"$\Delta E_{00}$")
ax.set_title("D65-FairFace7-ROI — per-trial ΔE00")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

# --- By ethnicity (histograms + mean/median bars) when >1 group ---
by_eth = {}
for r in rows:
    by_eth.setdefault((r.get("ethnicity") or "?").strip(), []).append(float(r["de00"]))
order = [e for e in ["Black", "Indian", "Asian", "Iranian", "White"] if e in by_eth]
order += [e for e in sorted(by_eth) if e not in order]

if len(order) >= 1:
    n_panels = len(order)
    ncols = min(3, n_panels)
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    vmax = max(max(v) for v in by_eth.values())
    bins = np.arange(0, max(16.5, np.ceil(vmax) + 1.5), 1.0)
    for i, eth in enumerate(order):
        ax = axes[i]
        v = by_eth[eth]
        ax.hist(v, bins=bins, color=colors_map.get(eth, "#7f8c8d"), edgecolor="white", alpha=0.9)
        ax.axvline(median(v), color="#e74c3c", ls="--", lw=1.4, label=f"med={median(v):.2f}")
        ax.axvline(5.55, color="#27ae60", ls=":", alpha=0.8)
        ax.set_title(f"{eth} (n={len(v)})")
        ax.set_xlabel(r"$\Delta E_{00}$"); ax.set_ylabel("Count")
        ax.legend(fontsize=8, frameon=False)
    for j in range(len(order), len(axes)):
        axes[j].axis("off")
    fig.suptitle(r"ΔE$_{00}$ by ethnicity", fontsize=13)
    plt.show()

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    x = np.arange(len(order)); w = 0.35
    ax.bar(x - w/2, [mean(by_eth[e]) for e in order], w, label="mean", color="#34495e")
    ax.bar(x + w/2, [median(by_eth[e]) for e in order], w, label="median", color="#e67e22")
    ax.axhline(5.55, color="#27ae60", ls="--", label="frozen 5.55")
    ax.axhline(5.93, color="#8e44ad", ls=":", label="camera-WB 5.93")
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel(r"$\Delta E_{00}$"); ax.set_title("Mean / median by ethnicity")
    ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

print(f"{'ID':8s} {'Eth':8s} {'ΔE00':>6s}  {'L*':>5s} {'a*':>5s} {'b*':>5s}")
for r in rows:
    print(f"{r['subject_id']:8s} {r.get('ethnicity',''):8s} {float(r['de00']):6.2f}  "
          f"{float(r['pipeline_L']):5.1f} {float(r['pipeline_a']):5.1f} {float(r['pipeline_b']):5.1f}")
print(f"{'ALL':8s} {'':8s} {mean(vals):6.2f}")

tsv = OUT_DIR / "table_emily_format.tsv"
if tsv.is_file():
    print("\nEmily TSV:\n", tsv.read_text())
    try:
        from google.colab import files
        files.download(str(tsv))
        files.download(str(OUT_DIR / "summary.json"))
    except Exception:
        pass


## 5 — Reference numbers

| Method | ROI | Mean ΔE₀₀ |
|---|---|---:|
| Camera-WB + affine | Apple | ~5.93 |
| Frozen `preawb_cat` 5500 K | Apple trimmed mean | **5.55** |
| MediaPipe cheek (older chart-free) | MediaPipe | ~6.36 |
| **D65-FairFace7-ROI** | Apple + FairFace-7 | **~3.63** |
| Demographics oracle ROI | Apple + true ethnicity | ~3.23 |

**Notes**
- Lab always comes from **linear RAW**; FairFace only chooses the cheek sampling rule.
- Full download (`LIMIT_PARTICIPANTS=0`) should approach mean ≈ 3.63 across ethnicities. `LIMIT_PARTICIPANTS=1` is Bryan-only (~3.4).
- Local full cohort: `python3 scripts/evaluate_pansor20_chartfree_d65.py --scr-mode preawb_cat --fixed-cat-k 5500 --l-sampling fairface7 --emily-tsv`
